In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import pyarrow as pa

In [ ]:
df = pd.read_csv('Heracleum mantegazzianum.csv', engine='pyarrow')
df_weather = pd.read_csv('knmi_weather_cache.csv', engine='pyarrow')
df_habitats = pd.read_csv('habitats_cbs_2022.csv', engine='pyarrow')

#df.sample(10)
#df_weather.sample(10)
#df_habitats.sample(10)

In [ ]:
df.info(verbose=True)
df.describe()

df_weather.info(verbose=True)
df_weather.describe()

df_habitats.info(verbose=True)
df_habitats.describe()

In [ ]:
#df[df.duplicated()] true

From the initial data description, it is visible that not all columns have the correct data types. The *eventDate* column should be formatted in **datetime** instead of in **string**, the *total_observations* column is formatted as **float**, and while that's usable, it should be of the **integer** type instead, and finally the *Heracleum mantegazzianum* column is in **string**, while that should be an **integer** column as well.

From the above duplicate check, it is also visible that there are **no** duplicates. (Ruben)

In [ ]:
for col in df.columns:
    uniques = df[col].unique()[:10].tolist()
    print(f"{col}: {uniques}...")
print(df.head(5).to_string())

TLDR;
some **NAN** values in total_observations
Heracleum mantegazzianum array has some *unknown* and *-1 negative values* (Mika)

In [ ]:
df['total_observations'] = pd.to_numeric(df['total_observations'], errors='coerce')
df['total_observations'] = df['total_observations'].fillna(0).astype(np.int32)
df['speciesgroup_observations'] = df['speciesgroup_observations'].astype(np.int32)
df['decimalLatitude'] = df['decimalLatitude'].astype(np.float32)
df['decimalLongitude'] = df['decimalLongitude'].astype(np.float32)
#print(df['total_observations'].dtype, len(df['total_observations']))
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_weather['mean_temp_c'] = df_weather['mean_temp_c'].astype(np.float32)
df_weather['max_temp_c'] = df_weather['max_temp_c'].astype(np.float32)
df_weather['min_temp_c'] = df_weather['min_temp_c'].astype(np.float32)
df_weather['precipitation_mm'] = df_weather['precipitation_mm'].astype(np.float32)
#df_weather['']

I observed that the total_observations column was with float values and NaNs. To mitigate this, I converted all NaNs to 0 and all values from float to int. (Marcell)

In [ ]:
cat_series = df['Heracleum mantegazzianum'].astype('category')
unique_cats = pd.to_numeric(cat_series.cat.categories, errors='coerce').to_numpy()
clean_cats = np.where(np.isnan(unique_cats) | (unique_cats < 0), 0, unique_cats).astype(np.int8)
df['Heracleum mantegazzianum'] = clean_cats[cat_series.cat.codes]

The Heracleum mantegazzianum column was originally in a string format. All unknown / NaN values were converted to 0 and all others including the new 0s were converted to integers. Furthermore, all values below 0 were raised to 0 as there was at least 1 instance where the row value was below 0 (-1). (Marcell)

In [ ]:
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')
#df['eventDate'].sample(10)

In [ ]:
df_merged = df.merge(df_weather, left_on='eventDate', right_on='date', how='left')
df_merged = df_merged.drop(columns='date')

# df_merged.info(verbose=True)

The *weather* and *observation* datasets were merged here. (Ruben)

The *eventDate* column was with *string* values instead of *datetime* values. To correct this, I changed the eventDate column to be of the ***datetime data type***. (Ruben)

In [ ]:
nl_layer = gpd.read_file("gadm41_NLD.gpkg", layer="ADM_ADM_1")
def show_distribution_map(df_map, nl):
    fig, ax = plt.subplots(figsize=(15, 10))
    nl.plot(ax=ax, color='lightgray', edgecolor='white')

    bin_w = 0.05
    x_bins = np.arange(df_map['decimalLongitude'].min(), df_map['decimalLongitude'].max() + bin_w, bin_w)
    y_bins = np.arange(df_map['decimalLatitude'].min(), df_map['decimalLatitude'].max() + bin_w, bin_w)

    heatmap, xedges, yedges = np.histogram2d(
        df_map['decimalLongitude'],
        df_map['decimalLatitude'],
        bins=[x_bins, y_bins],
        weights=df_map['total_observations']
    )

    heatmap = np.where(heatmap == 0, np.nan, heatmap)
    mesh = ax.pcolormesh(
        xedges,
        yedges,
        heatmap.T,
        alpha=0.6,
        cmap='viridis'
    )
    fig.colorbar(mesh, ax=ax, label='Total observations')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Distribution of total observations and their coordinate locations")
    plt.show()

show_distribution_map(df, nl_layer)

I took the latitude and longitude coordinates from the observations, and applied them in a *distplot* to show ***distribution of observations*** by using the *total_observations* column as a weight. This is done to learn where most observations are made. This means we have also learnt that some observations appear to have been made in ***Somalia***. Either that or the coordinates of those observations have been **inverted**. This is something to fix before we move forward.

We can either choose to *invert* these values and fix them, or choose to *remove them. Ideally, we shall do the former.

In [ ]:
df_nlmap = df[['decimalLatitude','decimalLongitude', 'total_observations']].copy()
mask = ((df_nlmap['decimalLatitude'] < 10) & (df_nlmap['decimalLongitude'] > 40))
df_nlmap.loc[mask, ['decimalLatitude', 'decimalLongitude']] = df_nlmap.loc[mask, ['decimalLongitude', 'decimalLatitude']].values

show_distribution_map(df_nlmap, nl_layer)

I *inverted* the coordinates so they can show up properly on the map. However, now we can see that someone was trying to be funny and added a random 67 to the map. We will need to remove this.

In [ ]:
df = df[
    (df['decimalLatitude'].between(50.7, 53.6)) &
    (df['decimalLongitude'].between(3.3, 7.2))]

show_distribution_map(df, nl_layer)

Here the 67 was *removed* from the map.

In [ ]:
df_corr = df_merged.corr(method='spearman', numeric_only=True)
mask = np.triu(np.ones_like(df_corr, dtype=bool))
sns.heatmap(df_corr, annot=True, mask=mask)

Upon creating a quick correlation matrix we can conclude there's some correlation between total_observations and speciesgroup_observations as well as minor correlation between speciesgroup_observation and our target value. There is also high correlation between latitude and longitude. This makes perfect sense.

In [ ]:
df_months = df.groupby(df['eventDate'].dt.to_period('M'))['Heracleum mantegazzianum'].sum()
fig, ax = plt.subplots(figsize=(10, 6))
df_months.plot(kind='bar', ax=ax, width=0.8, color='#4c72b0', edgecolor='none')

ax.set_xticks([])

ax.set_xlabel("eventDate grouped by month starting at january 2010")
ax.set_ylabel("Heracleum mantegazzianum")
ax.grid(axis='y', linestyle='-', alpha=0.3)
ax.set_axisbelow(True)
plt.show()

In [ ]:
win: str = "Winter"
spr: str = "Spring"
smr: str = "Summer"
aut: str = "Autumn"

season_map = {
    12: win, 1: win, 2: win,
    3: spr, 4: spr, 5: spr,
    6: smr, 7: smr, 8: smr,
    9: aut, 10: aut, 11: aut }

df_months = df[['eventDate', 'Heracleum mantegazzianum']].copy()
df_months['eventDate'] = df_months['eventDate'].dt.month.map(season_map)
df_months = df_months.groupby('eventDate').sum(numeric_only=True)

sns.barplot(data=df_months, x='eventDate', y='Heracleum mantegazzianum')

In [ ]:
bias_df = df[['total_observations', 'speciesgroup_observations', 'Heracleum mantegazzianum']]
bias_df = bias_df.sum().reset_index()
bias_df.columns = ['Observation type', 'Total count']

label_map = {
    'total_observations': 'Total Observations',
    'speciesgroup_observations': 'Species Group Observations',
    'Heracleum mantegazzianum': 'Heracleum Mantegazzianum'
}

bias_df['Observation type'] = bias_df['Observation type'].map(label_map)
totals = sorted(bias_df['Total count'].tolist())

plt.figure(figsize=(8, 6))
ax = sns.barplot(data=bias_df, x='Observation type', y='Total count')
ax.set_yscale('log')
plt.yticks(totals, [f"{int(val):,}" for val in totals])
plt.grid(axis='y', linestyle='--', alpha=0.7, color='red')
plt.tight_layout()
plt.show()